In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../Data/Raw/BankChurners.csv")

In [3]:
df.shape

(10127, 23)

In [4]:
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10127 entries, 0 to 10126
Data columns (total 23 columns):
 #   Column                                                                                                                              Non-Null Count  Dtype  
---  ------                                                                                                                              --------------  -----  
 0   CLIENTNUM                                                                                                                           10127 non-null  int64  
 1   Attrition_Flag                                                                                                                      10127 non-null  str    
 2   Customer_Age                                                                                                                        10127 non-null  int64  
 3   Gender                                                                                      

In [5]:
df.columns.tolist()

['CLIENTNUM',
 'Attrition_Flag',
 'Customer_Age',
 'Gender',
 'Dependent_count',
 'Education_Level',
 'Marital_Status',
 'Income_Category',
 'Card_Category',
 'Months_on_book',
 'Total_Relationship_Count',
 'Months_Inactive_12_mon',
 'Contacts_Count_12_mon',
 'Credit_Limit',
 'Total_Revolving_Bal',
 'Avg_Open_To_Buy',
 'Total_Amt_Chng_Q4_Q1',
 'Total_Trans_Amt',
 'Total_Trans_Ct',
 'Total_Ct_Chng_Q4_Q1',
 'Avg_Utilization_Ratio',
 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']

In [6]:
df.isna().sum()

CLIENTNUM                                                                                                                             0
Attrition_Flag                                                                                                                        0
Customer_Age                                                                                                                          0
Gender                                                                                                                                0
Dependent_count                                                                                                                       0
Education_Level                                                                                                                       0
Marital_Status                                                                                                                        0
Income_Category                                 

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df["CLIENTNUM"].nunique()

10127

In [9]:
len(df)

10127

In [10]:
df["Attrition_Flag"].value_counts()

Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64

In [11]:
counts = df["Attrition_Flag"].value_counts()
counts / counts.sum()

Attrition_Flag
Existing Customer    0.83934
Attrited Customer    0.16066
Name: count, dtype: float64

In [12]:
print(df.describe())

          CLIENTNUM  Customer_Age  Dependent_count  Months_on_book  \
count  1.012700e+04  10127.000000     10127.000000    10127.000000   
mean   7.391776e+08     46.325960         2.346203       35.928409   
std    3.690378e+07      8.016814         1.298908        7.986416   
min    7.080821e+08     26.000000         0.000000       13.000000   
25%    7.130368e+08     41.000000         1.000000       31.000000   
50%    7.179264e+08     46.000000         2.000000       36.000000   
75%    7.731435e+08     52.000000         3.000000       40.000000   
max    8.283431e+08     73.000000         5.000000       56.000000   

       Total_Relationship_Count  Months_Inactive_12_mon  \
count              10127.000000            10127.000000   
mean                   3.812580                2.341167   
std                    1.554408                1.010622   
min                    1.000000                0.000000   
25%                    3.000000                2.000000   
50%            

## Data Cleaning Phase

Steps:

- Remove Naive Bayes columns from the dataset to prevent data leakage
- Standardize the column names
- Ensure customer identifiers are not treated as numeric data
- Create a churn indicator for analytical use represented in binary
- Avoid imputing values, preserve unknown values instead

In [13]:
naive_columns = []
for col in df.columns:
    if "Naive_Bayes_Classifier" in col:
        naive_columns.append(col)

df = df.drop(columns=naive_columns)
df.shape

(10127, 21)

In [14]:
df.columns = df.columns.str.strip().str.lower()

df.columns.tolist()

['clientnum',
 'attrition_flag',
 'customer_age',
 'gender',
 'dependent_count',
 'education_level',
 'marital_status',
 'income_category',
 'card_category',
 'months_on_book',
 'total_relationship_count',
 'months_inactive_12_mon',
 'contacts_count_12_mon',
 'credit_limit',
 'total_revolving_bal',
 'avg_open_to_buy',
 'total_amt_chng_q4_q1',
 'total_trans_amt',
 'total_trans_ct',
 'total_ct_chng_q4_q1',
 'avg_utilization_ratio']

In [15]:
replacement_names = {
    "clientnum": "customer_id",
    "attrition_flag": "attrition_status",
    "months_on_book": "customer_tenure_months"
}

df = df.rename(columns=replacement_names)

df.columns.tolist()

['customer_id',
 'attrition_status',
 'customer_age',
 'gender',
 'dependent_count',
 'education_level',
 'marital_status',
 'income_category',
 'card_category',
 'customer_tenure_months',
 'total_relationship_count',
 'months_inactive_12_mon',
 'contacts_count_12_mon',
 'credit_limit',
 'total_revolving_bal',
 'avg_open_to_buy',
 'total_amt_chng_q4_q1',
 'total_trans_amt',
 'total_trans_ct',
 'total_ct_chng_q4_q1',
 'avg_utilization_ratio']

In [16]:
df["customer_id"] = df["customer_id"].astype("string")

df["customer_id"].dtype

<StringDtype(storage='python', na_value=<NA>)>

In [17]:
df["churned"] = (df["attrition_status"] == "Attrited Customer").astype(int)

df[["attrition_status", "churned"]].value_counts()

attrition_status   churned
Existing Customer  0          8500
Attrited Customer  1          1627
Name: count, dtype: int64

In [18]:
inspect_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in inspect_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


customer_id
customer_id
768805383    1
818770008    1
713982108    1
769911858    1
709106358    1
            ..
772366833    1
710638233    1
716506083    1
717406983    1
714337233    1
Name: count, Length: 10127, dtype: Int64

attrition_status
attrition_status
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64

gender
gender
F    5358
M    4769
Name: count, dtype: int64

education_level
education_level
Graduate         3128
High School      2013
Unknown          1519
Uneducated       1487
College          1013
Post-Graduate     516
Doctorate         451
Name: count, dtype: int64

marital_status
marital_status
Married     4687
Single      3943
Unknown      749
Divorced     748
Name: count, dtype: int64

income_category
income_category
Less than $40K    3561
$40K - $60K       1790
$80K - $120K      1535
$60K - $80K       1402
Unknown           1112
$120K +            727
Name: count, dtype: int64

card_category
card_category
Blue        9436
Silver       5

In [19]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())
print("Missing values:", df.isna().sum().sum())

Rows: 10127
Columns: 22
Duplicate rows: 0
Duplicate customer IDs: 0
Missing values: 0


In [20]:
print("Invalid ages:",
      (~df["customer_age"].between(18, 100)).sum())

print("Invalid inactivity months:",
      (~df["months_inactive_12_mon"].between(0, 12)).sum())

print("Invalid relationship counts:",
      (df["total_relationship_count"] < 1).sum())

print("Negative credit limits:",
      (df["credit_limit"] < 0).sum())

print("Negative transaction amounts:",
      (df["total_trans_amt"] < 0).sum())

print("Invalid utilization ratios:",
      (~df["avg_utilization_ratio"].between(0, 1)).sum())

Invalid ages: 0
Invalid inactivity months: 0
Invalid relationship counts: 0
Negative credit limits: 0
Negative transaction amounts: 0
Invalid utilization ratios: 0


In [21]:
open_to_buy_difference = (
    df["credit_limit"]
    - df["total_revolving_bal"]
    - df["avg_open_to_buy"]
).abs()

open_to_buy_difference.max()

np.float64(5.684341886080802e-14)

In [22]:
calculated_utilization = (
    df["total_revolving_bal"] / df["credit_limit"]
)

utilization_difference = (
    df["avg_utilization_ratio"] - calculated_utilization
).abs()

utilization_difference.describe()

count    10127.000000
mean         0.000188
std          0.000165
min          0.000000
25%          0.000004
50%          0.000167
75%          0.000333
max          0.000500
dtype: float64

In [23]:
df.to_csv(
    "../Data/Processed/BankChurners_Clean.csv",
    index=False
)

## Data Cleaning Summary

The raw data contained 10,127 records and 23 columns. The cleaned data contains 10,127 records and 22 columns.

- Validated uniqueness and missing values
- Removed Naive Bayes columns to prevent data leakage
- Standardized column names
- Change customer identifier into a string data type
- Created a binary variable for churn status
- Validated consistency